In [1]:

from dataclasses import dataclass
from typing import List, Optional, Tuple
import numpy as np
import pandas as pd


class BigMNumber:
    """Biểu diễn số học dạng: α*M + β phục vụ phương pháp Big-M."""

    def __init__(self, m: float = 0.0, c: float = 0.0) -> None:
        self.m = float(m)
        self.c = float(c)

    def __add__(self, other):
        if isinstance(other, (int, float)):
            return BigMNumber(self.m, self.c + other)
        return BigMNumber(self.m + other.m, self.c + other.c)

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        if isinstance(other, (int, float)):
            return BigMNumber(self.m, self.c - other)
        return BigMNumber(self.m - other.m, self.c - other.c)

    def __rsub__(self, other):
        if isinstance(other, (int, float)):
            return BigMNumber(-self.m, other - self.c)
        return BigMNumber(other.m - self.m, other.c - self.c)

    def __mul__(self, scalar: float):
        return BigMNumber(self.m * scalar, self.c * scalar)

    def __rmul__(self, scalar: float):
        return self.__mul__(scalar)

    def __gt__(self, other):
        o_m = other.m if isinstance(other, BigMNumber) else 0.0
        o_c = other.c if isinstance(other, BigMNumber) else float(other)
        if not np.isclose(self.m, o_m, atol=1e-8):
            return self.m > o_m
        return self.c > o_c + 1e-8

    def is_positive(self) -> bool:
        return self > 0.0

    def to_formatted_str(self, decimals: int = 5) -> str:
        """Định dạng số theo số chữ số thập phân yêu cầu."""
        m_part = ""
        if not np.isclose(self.m, 0.0, atol=1e-8):
            m_round = round(self.m, decimals)
            if np.isclose(m_round, 1.0):
                m_part = "M"
            elif np.isclose(m_round, -1.0):
                m_part = "-M"
            elif m_round.is_integer():
                m_part = f"{int(m_round)}M"
            else:
                m_part = f"{m_round:.{decimals}f}M"

        c_round = round(self.c, decimals)
        c_str = f"{c_round:.{decimals}f}"

        if not m_part:
            return c_str
        if np.isclose(c_round, 0.0, atol=1e-8):
            return m_part
        if c_round > 0:
            return f"{m_part}+{c_str}"
        return f"{m_part}{c_str}"

    def __str__(self) -> str:
        return self.to_formatted_str(decimals=5)


@dataclass
class SimplexResult:
    status: str
    optimal_x: Optional[np.ndarray] = None
    optimal_value: Optional[float] = None
    iterations: int = 0


class PandasBigMSimplex:
    """Bộ giải Big-M Simplex hỗ trợ định dạng số thập phân thứ 5."""

    def __init__(
        self,
        c: List[BigMNumber],
        A: np.ndarray,
        b: np.ndarray,
        basis_vars: List[int],
        var_names: List[str],
        drop_leaving_artificial: bool = True,
        precision: int = 5,
    ) -> None:
        self.c = list(c)
        self.A = np.array(A, dtype=np.float64)
        self.b = np.array(b, dtype=np.float64)
        self.basis_vars = list(basis_vars)
        self.var_names = list(var_names)
        self.num_constraints, self.num_vars = self.A.shape
        self.drop_leaving_artificial = drop_leaving_artificial
        self.precision = precision
        self.active_cols = list(range(self.num_vars))

    def _calculate_deltas(self) -> Tuple[List[BigMNumber], BigMNumber]:
        c_B = [self.c[idx] for idx in self.basis_vars]
        c_B_m = np.array([cb.m for cb in c_B])
        c_B_c = np.array([cb.c for cb in c_B])

        f_val = BigMNumber(
            m=float(np.dot(c_B_m, self.b)),
            c=float(np.dot(c_B_c, self.b))
        )

        term_m = np.dot(c_B_m, self.A)
        term_c = np.dot(c_B_c, self.A)
        deltas = [
            BigMNumber(m= - term_m[j] + self.c[j].m, c= - term_c[j] + self.c[j].c)
            for j in range(self.num_vars)
        ]
        return deltas, f_val

    def _build_dataframe(
        self,
        deltas: List[BigMNumber],
        f_val: BigMNumber,
        lambdas: Optional[np.ndarray] = None,
    ) -> pd.DataFrame:
        p = self.precision
        rows = []
        for i in range(self.num_constraints):
            row = {
                "Hệ số": self.c[self.basis_vars[i]].to_formatted_str(p),
                "Cơ sở": self.var_names[self.basis_vars[i]],
                "Phương án": f"{self.b[i]:.{p}f}",
            }
            for j in self.active_cols:
                row[self.var_names[j]] = f"{self.A[i, j]:.{p}f}"
            if lambdas is not None:
                row["λ"] = f"{lambdas[i]:.{p}f}" if not np.isnan(lambdas[i]) else "-"
            rows.append(row)

        delta_row = {
            "Hệ số": "",
            "Cơ sở": "Δ_j",
            "Phương án": f_val.to_formatted_str(p),
        }
        for j in self.active_cols:
            delta_row[self.var_names[j]] = deltas[j].to_formatted_str(p)
        if lambdas is not None:
            delta_row["λ"] = ""
        rows.append(delta_row)

        return pd.DataFrame(rows)

    def solve(self, max_iters: int = 15, verbose: bool = True) -> SimplexResult:
        pd.set_option("display.width", 1000)
        pd.set_option("display.max_columns", None)

        for step in range(max_iters):
            deltas, f_val = self._calculate_deltas()
            positive_deltas = [j for j in self.active_cols if deltas[j].is_positive()]

            if not positive_deltas:
                if verbose:
                    print(f"\n{'='*35} BẢNG ĐƠN HÌNH {step} (TỐI ƯU) {'='*35}")
                    print(self._build_dataframe(deltas, f_val).to_string(index=False))
                    print("\n[+] ĐÃ ĐẠT ĐIỀU KIỆN TỐI ƯU (Tất cả Δ_j <= 0).")

                optimal_x = np.zeros(self.num_vars)
                for i, idx in enumerate(self.basis_vars):
                    optimal_x[idx] = self.b[i]

                return SimplexResult(
                    status="optimal",
                    optimal_x=optimal_x,
                    optimal_value=f_val.c,
                    iterations=step,
                )

            # Chọn cột chủ yếu p
            p = positive_deltas[0]
            for j in positive_deltas[1:]:
                if deltas[j] > deltas[p]:
                    p = j

            # Tính λ_i
            lambdas = np.full(self.num_constraints, np.nan)
            valid_rows = np.where(self.A[:, p] > 1e-8)[0]

            if len(valid_rows) == 0:
                if verbose:
                    print(f"\n[-] Cột {self.var_names[p]} không có hệ số a_ip > 0. Bài toán không bị chặn!")
                return SimplexResult(status="unbounded", iterations=step)

            lambdas[valid_rows] = self.b[valid_rows] / self.A[valid_rows, p]

            if verbose:
                print(f"\n{'='*40} BẢNG ĐƠN HÌNH {step} {'='*40}")
                print(self._build_dataframe(deltas, f_val, lambdas).to_string(index=False))

            # Chọn dòng chủ yếu q
            q = int(valid_rows[np.argmin(lambdas[valid_rows])])
            leaving_var_idx = self.basis_vars[q]
            pivot_val = self.A[q, p]

            if verbose:
                print(f"[>] Ẩn đưa vào: {self.var_names[p]} (Δ = {deltas[p].to_formatted_str(self.precision)})")
                print(f"[<] Ẩn đưa ra: {self.var_names[leaving_var_idx]} (min λ = {lambdas[q]:.{self.precision}f})")
                print(f"[*] Phần tử trục: {pivot_val:.{self.precision}f}")

            # Phép khử Gauss-Jordan
            self.A[q] /= pivot_val
            self.b[q] /= pivot_val

            factors = self.A[:, p].copy()
            factors[q] = 0.0

            self.A -= factors[:, np.newaxis] * self.A[q]
            self.b -= factors * self.b[q]

            self.basis_vars[q] = p

            if self.drop_leaving_artificial and self.c[leaving_var_idx].m > 0:
                if leaving_var_idx in self.active_cols:
                    self.active_cols.remove(leaving_var_idx)
                    if verbose:
                        print(f"[-] Ẩn nhân tạo {self.var_names[leaving_var_idx]} đã rời cơ sở -> Không hiển thị ở bảng sau.")

        raise TimeoutError("Đạt giới hạn vòng lặp tối đa.")


In [2]:
def main():
    var_names = ["x1", "x2", "x3", "x4", "x5"]

    # Hệ số hàm mục tiêu min f(x)
    c = [
        BigMNumber(m=0, c=-2),
        BigMNumber(m=0, c=6),
        BigMNumber(m=0, c=4),
        BigMNumber(m=0, c=-2),
        BigMNumber(m=0, c=3),   # x5 (ẩn phụ)

    ]

    # Ma trận A tại Bảng 0
    A = np.array([
        [ 1.0, 2.0, 4.0, 0.0, 0.0],
        [ 0.0, 4.0, 2.0, 1.0, 0.0],
        [ 0.0, 3.0, 0.0, 0.0, 1.0],
    ])

    # Cột phương án b
    b = np.array([312.0, 360.0, 216.0])

    # Cơ sở xuất phát: x6, x5, x7
    basis_vars = [0, 3, 4]

    solver = PandasBigMSimplex(
        c, A, b, basis_vars, var_names, drop_leaving_artificial=True
    )
    res = solver.solve()

    if res.status == "optimal":
        print("\n" + "=" * 25 + " NGHIỆM TỐI ƯU " + "=" * 25)
        for name, val in zip(var_names[:5], res.optimal_x[:5]):
            print(f"  {name} = {val:.5f}")
        print(f"  Giá trị tối ưu f_min = {res.optimal_value:.5f}")


if __name__ == "__main__":
    main()


======================================== BẢNG ĐƠN HÌNH 0 ========================================
   Hệ số Cơ sở  Phương án      x1      x2       x3      x4      x5         λ
-2.00000    x1  312.00000 1.00000 2.00000  4.00000 0.00000 0.00000  78.00000
-2.00000    x4  360.00000 0.00000 4.00000  2.00000 1.00000 0.00000 180.00000
 3.00000    x5  216.00000 0.00000 3.00000  0.00000 0.00000 1.00000         -
           Δ_j -696.00000 0.00000 9.00000 16.00000 0.00000 0.00000          
[>] Ẩn đưa vào: x3 (Δ = 16.00000)
[<] Ẩn đưa ra: x1 (min λ = 78.00000)
[*] Phần tử trục: 4.00000

======================================== BẢNG ĐƠN HÌNH 1 ========================================
   Hệ số Cơ sở Phương án       x1      x2      x3      x4      x5         λ
 4.00000    x3  78.00000  0.25000 0.50000 1.00000 0.00000 0.00000 156.00000
-2.00000    x4 204.00000 -0.50000 3.00000 0.00000 1.00000 0.00000  68.00000
 3.00000    x5 216.00000  0.00000 3.00000 0.00000 0.00000 1.00000  72.00000
           Δ_j 5

In [3]:
def main():
    var_names = ["x1", "x2'", "x31", "x32", "x4", "x5","x6", "x7"]

    # Hệ số hàm mục tiêu min f(x)
    c = [
        BigMNumber(m=0, c=1),
        BigMNumber(m=0, c=2),
        BigMNumber(m=0, c=3),
        BigMNumber(m=0, c=-3),
        BigMNumber(m=0, c=-4),   # x5 (ẩn phụ)
        BigMNumber(m=0, c=0),
        BigMNumber(m=-1, c=0),   # x7 (ẩn nhân tạo M)
        BigMNumber(m=0, c=0),   # x8 (ẩn nhân tạo M)
    ]

    # Ma trận A tại Bảng 0
    A = np.array([
        [ 2.0, 3.0, 1.0, -1.0, -1.0, 1.0, 0.0, 0],
        [ -2.0, -1.0, 3.0, -3.0, -3.0, 0.0, 1.0, 0.0],
        [ -1.0, 3.0, 1.0, -1.0, -1.0, 0.0, 0.0, 1.0],
    ])

    # Cột phương án b
    b = np.array([ 10, 12, 11])

    # Cơ sở xuất phát: x6, x5, x7
    basis_vars = [5, 6, 7]

    solver = PandasBigMSimplex(
        c, A, b, basis_vars, var_names, drop_leaving_artificial=True
    )
    res = solver.solve()

    if res.status == "optimal":
        print("\n" + "=" * 25 + " NGHIỆM TỐI ƯU " + "=" * 25)
        for name, val in zip(var_names[:5], res.optimal_x[:5]):
            print(f"  {name} = {val:.5f}")
        print(f"  Giá trị tối ưu f_min = {res.optimal_value:.5f}")


if __name__ == "__main__":
    main()


======================================== BẢNG ĐƠN HÌNH 0 ========================================
  Hệ số Cơ sở Phương án          x1        x2'        x31         x32          x4      x5      x6      x7        λ
0.00000    x5  10.00000     2.00000    3.00000    1.00000    -1.00000    -1.00000 1.00000 0.00000 0.00000 10.00000
     -M    x6  12.00000    -2.00000   -1.00000    3.00000    -3.00000    -3.00000 0.00000 1.00000 0.00000  4.00000
0.00000    x7  11.00000    -1.00000    3.00000    1.00000    -1.00000    -1.00000 0.00000 0.00000 1.00000 11.00000
          Δ_j      -12M -2M+1.00000 -M+2.00000 3M+3.00000 -3M-3.00000 -3M-4.00000 0.00000 0.00000 0.00000         
[>] Ẩn đưa vào: x31 (Δ = 3M+3.00000)
[<] Ẩn đưa ra: x6 (min λ = 4.00000)
[*] Phần tử trục: 3.00000

======================================== BẢNG ĐƠN HÌNH 1 ========================================
  Hệ số Cơ sở Phương án       x1      x2'     x31      x32       x4      x5         x6      x7       λ
0.00000    x5   6.00000  

In [3]:
def main():
    var_names = ["x1", "x2'", "x3", "x4", "x5","x6"]

    # Hệ số hàm mục tiêu min f(x)
    c = [
        BigMNumber(m=0, c=2),
        BigMNumber(m=0, c=4),
        BigMNumber(m=0, c=3),
        BigMNumber(m=0, c=0),
        BigMNumber(m=0, c=0),
        BigMNumber(m=0, c=0),

    ]

    # Ma trận A tại Bảng 0
    A = np.array([
        [ 3.0, 4.0, 2.0, 1.0, 0.0, 0.0],
        [ 4.0, 1.0, 2.0, 0.0, 1.0, 0.0],
        [ 1.0, 3.0, 2.0, 0.0, 0.0, 1.0],
    ])

    # Cột phương án b
    b = np.array([ 180, 120, 240])

    # Cơ sở xuất phát: x6, x5, x7
    basis_vars = [3, 4, 5]

    solver = PandasBigMSimplex(
        c, A, b, basis_vars, var_names, drop_leaving_artificial=True
    )
    res = solver.solve()

    if res.status == "optimal":
        print("\n" + "=" * 25 + " NGHIỆM TỐI ƯU " + "=" * 25)
        for name, val in zip(var_names[:5], res.optimal_x[:5]):
            print(f"  {name} = {val:.5f}")
        print(f"  Giá trị tối ưu f_min = {res.optimal_value:.5f}")


if __name__ == "__main__":
    main()


======================================== BẢNG ĐƠN HÌNH 0 ========================================
  Hệ số Cơ sở Phương án      x1     x2'      x3      x4      x5      x6         λ
0.00000    x4 180.00000 3.00000 4.00000 2.00000 1.00000 0.00000 0.00000  45.00000
0.00000    x5 120.00000 4.00000 1.00000 2.00000 0.00000 1.00000 0.00000 120.00000
0.00000    x6 240.00000 1.00000 3.00000 2.00000 0.00000 0.00000 1.00000  80.00000
          Δ_j   0.00000 2.00000 4.00000 3.00000 0.00000 0.00000 0.00000          
[>] Ẩn đưa vào: x2' (Δ = 4.00000)
[<] Ẩn đưa ra: x4 (min λ = 45.00000)
[*] Phần tử trục: 4.00000

======================================== BẢNG ĐƠN HÌNH 1 ========================================
  Hệ số Cơ sở Phương án       x1     x2'      x3       x4      x5      x6         λ
4.00000   x2'  45.00000  0.75000 1.00000 0.50000  0.25000 0.00000 0.00000  90.00000
0.00000    x5  75.00000  3.25000 0.00000 1.50000 -0.25000 1.00000 0.00000  50.00000
0.00000    x6 105.00000 -1.25000 0.00000 0.